# Notebook 2: Data Cleansing

## Goal
Transform the raw ONS Excel file into clean, analysis-ready CSV files.

## Tables being cleaned
- **Table 1**: UK total population 1851-2024 (simple cleaning)
- **Table 3**: UK population by sex + single year of age 1971-2024 (requires shaping)

In [1]:
import pandas as pd
import numpy as np

RAW_FILE= 'data/raw/ukpopulationestimates18382024.xlsx'
CLEAN_DIR = 'data/clean/'

In [2]:
df_t1_raw = pd.read_excel(RAW_FILE, sheet_name='Table 1', header=1)

df_t1 = df_t1_raw[['Year', 'Total Population']].copy()
df_t1 = df_t1.dropna(subset=['Year', 'Total Population'])

df_t1['Year'] = (
    df_t1['Year']
    .astype(str)
    .str.replace('Mid-', '')
    .str.strip()
    .astype(int)
)

df_t1 = df_t1.sort_values('Year').reset_index(drop=True)

print("Table 1 cleaned shape:", df_t1.shape)
print(df_t1.head())
print(df_t1.tail())

df_t1.to_csv(f"{CLEAN_DIR}table1_uk_population.csv", index=False)
print("\nSaved: table1_uk_total_population.csv")

Table 1 cleaned shape: (156, 2)
   Year  Total Population
0  1851          27368800
1  1861          28917900
2  1871          31484700
3  1872          31874200
4  1873          32177600
     Year  Total Population
151  2020          66739867
152  2021          66977988
153  2022          67636134
154  2023          68526183
155  2024          69281437

Saved: table1_uk_total_population.csv


In [3]:
df_t3_raw = pd.read_excel(RAW_FILE, sheet_name='Table 3', header=1)
df_t3 = df_t3_raw.copy()

df_t3 = df_t3[df_t3['Age'] != 'All Ages']
df_t3['Age'] = pd.to_numeric(df_t3['Age'], errors='coerce').astype(int)
df_t3['Sex'] = df_t3['Sex'].str.strip()

year_cols = [c for c in df_t3.columns if str(c).startswith('Mid-')]
print (f"Year columns found: {len(year_cols)} ({year_cols[-1]} to {year_cols[0]})")

df_long = df_t3,melt(
    id_vars=['Age', 'Sex'],
    value_vars=year_cols,
    var_name='Year_Label',
var_name='Population'
    
)

df_long['Year'] = df_long['Year_Label'].str.replace('Mid-', '').astype(int)
df_long = df_long.drop(columns='Year_Label')

df_long['Population'] = pd.to_numeric(df_long['Population'], errors='coerce')
df_long = df_long.dropna(subset=['Population'])
df_long['Population'] = df_long['Population'].astype(int)

df_long = df_long.sort_values(['Year', 'Sex', 'Age']).reset_index(drop=True)

print("\nCleaned long-format shape:", df_long.shape)
print("\nSex values:", df_long['Sex'].unique())
print("Age range:", df_long['Age'].min(), "to", df_long['Age'].max())
print("Year range:", df_long['Year'].min(), "to", df_long['Year'].max())
print("\nFirst 10 rows:")
print(df_long.head(10))

df_long.to.csv(f'{CLEAN_DIR}table3_uk_population_long.csv', index=False)
print("\nSaved: table3_uk_population_long.csv")

IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer

In [ ]:
df_t3 = df_t3_raw.copy()

# Remove "All Ages"
df_t3 = df_t3[df_t3['Age'] != 'All Ages']

# Convert Age to numeric, but DO NOT convert int yet
df_t3['Age'] = pd.to_numeric(df_t3['Age'], errors='coerce')

# Drop rows where Age became NaN
df_t3 = df_t3.dropna(subset=['Age'])

# Now safely convert to int
df_t3['Age'] = df_t3['Age'].astype(int)

# Clean Sex column
df_t3['Sex'] = df_t3['Sex'].str.strip()

In [ ]:
df_t3['Age'].isna().sum()

In [ ]:
year_cols = [c for c in df_t3.columns if str(c).startswith('Mid-')]
print(f"Found {len(year_cols)} year columns")
print(year_cols[:5], "...", year_cols[-5:])

In [ ]:
df_long = df_t3.melt(
    id_vars=['Age', 'Sex'],
    value_vars=year_cols,
    var_name='Year_Label',
    value_name='Population'
)

In [ ]:
df_long['Year'] = df_long['Year_Label'].str.replace('Mid-', '').astype(int)
df_long = df_long.drop(columns='Year_Label')

In [ ]:
df_long['Population'] = pd.to_numeric(df_long['Population'], errors='coerce')
df_long = df_long.dropna(subset=['Population'])
df_long['Population'] = df_long['Population'].astype(int)

In [ ]:
def age_group(age):
    if age <= 17:
        return '0-17 Children'
    elif age <= 64:
        return '18-64 Working age'
    else:
        return '65+ Older adults'

df_long['Age_Group'] = df_long['Age'].apply(age_group)

In [ ]:
df_long['Age'].apply(type).value_counts()

In [ ]:
df_long['Age'] = pd.to_numeric(df_long['Age'], errors='coerce')

In [ ]:
df_long = df_long.dropna(subset=['Age'])

In [ ]:
df_long['Age'] = df_long['Age'].astype(int)

In [ ]:
df_long['Age_Group'] = df_long['Age'].apply(age_group)

In [ ]:
df_long.to_csv(f'{CLEAN_DIR}table3_uk_population_long.csv', index=False)

In [ ]:
print("=== DATA QUALITY REPORT ===\n")
print(f"Total rows: {len(df_long):,}")
print(f"Missing values: {df_long.isnull().sum().sum()}")
print(f"Duplicate rows: {df_long.duplicated().sum()}")
print(f"Negative population values: {(df_long['Population'] < 0).sum()}")
print(f"Year range: {df_long['Year'].min()} to {df_long['Year'].max()}")
print(f"Age range: {df_long['Age'].min()} to {df_long['Age'].max()}")
print(f"Sex category: {sorted(df_long['Sex'].unique())}")
print(f"\nPopulation stats:")
print(df_long['Population'].describe())

## Notebook 2 - Data Cleansing Complete
This notebook has successfully completely all required data-cleaning steps:
Loaded the raw ONS Excel workbook
Cleaned Table 1 (UK total population, 1851-2024)
Cleaned and reshaped Table 3 (population by sex and single year of age, 1971-2024)
Converted Table 3 from wide to long format
Standardised column names and data types
Added the Age_Group classification
Performed full data quality checks
Exported final tidy CSV files to the clean/ directory
## This dataset is not complete